# Silver → Gold Data & Analytics Preparation (Azure Synapse PySpark)

Notebook ini membaca 5 dataset bersih dari container **`silver`** pada ADLS Gen2 (`georisk`) dan membangun **8 tabel Gold** di container **`gold`** untuk **Web/Dashboard Operasional**, **Agregasi Analitik (BI/KPI/Heatmap)**, dan **Machine Learning Prediksi Karhutla**:

### A. Serving Tables (Web & Dashboard Real-Time)
1. `gold/dashboard_air_quality_weather/` — Kualitas udara + cuaca per jam per kota.
2. `gold/dashboard_volcano_status/` — Status terkini tiap gunung api (level 1–4 & high alert).
3. `gold/dashboard_fire_hotspots/` — Titik panas FIRMS + intensitas FRP + irisan tile MODIS MCD64A1.

### B. Analytical Aggregation Tables (KPI, Scorecard, Heatmap)
4. `gold/analytics_daily_national_kpi/` — KPI harian nasional (total hotspot, total FRP, moving average 7 hari, perubahan DoD, rata-rata AQI/PM2.5, jumlah kota tidak sehat).
5. `gold/analytics_city_risk_scorecard/` — Scorecard risiko multi-bahaya (0–100) per kota/provinsi per hari (skor karhutla + skor kualitas udara + arah angin dominan).
6. `gold/analytics_volcano_summary/` — Agregasi jumlah gunung api per level aktivitas & daftar gunung Siaga/Awas per provinsi.
7. `gold/analytics_spatial_hotspot_grid/` — Agregasi heatmap spasial per grid 0.25° per bulan (hotspot, FRP, hari aktif, overlap burned scar).

### C. Machine Learning Feature Store
8. `gold/ml_karhutla_features/` — Feature store harian per `(location, date)` siap training & prediksi ML karhutla.

In [ ]:
# Parameter cell (dapat di-override oleh Azure Synapse Pipeline)
storage_account = "georisk"
silver_container = "silver"
gold_container = "gold"
hotspot_radius_km = 50.0

In [ ]:
import json
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window

SILVER_ROOT = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"
GOLD_ROOT = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net"


def path_exists(path: str) -> bool:
    try:
        return bool(mssparkutils.fs.ls(path))
    except Exception:
        return False


def write_gold_parquet(
    df: DataFrame,
    table_name: str,
    partition_cols: list[str] | None = None,
) -> int:
    """Tulis dataset Gold sebagai Parquet secara aman via staging swap."""
    target_path = f"{GOLD_ROOT}/{table_name}"
    staging_path = f"{GOLD_ROOT}/_staging_{table_name}"

    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(staging_path)

    if path_exists(target_path):
        mssparkutils.fs.rm(target_path, True)
    mssparkutils.fs.mv(staging_path, target_path)

    count = spark.read.parquet(target_path).count()
    print(f"[gold/{table_name}] written -> {count} rows at {target_path}")
    return count


# Baca semua 5 tabel Silver
df_firms = spark.read.parquet(f"{SILVER_ROOT}/firms")
df_magma = spark.read.parquet(f"{SILVER_ROOT}/magma")
df_aq = spark.read.parquet(f"{SILVER_ROOT}/air_quality")
df_weather = spark.read.parquet(f"{SILVER_ROOT}/weather")
df_mcd = spark.read.parquet(f"{SILVER_ROOT}/mcd_burned_scar")

gold_summary = {"gold_counts": {}}

In [ ]:
# 1. Gold Serving: Kualitas Udara + Cuaca per Jam per Kota
aq_sel = df_aq.select(
    "location",
    "province",
    "latitude",
    "longitude",
    "timestamp_utc",
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "us_aqi",
)
weather_sel = df_weather.select(
    "location",
    "timestamp_utc",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
)

w_latest_city = Window.partitionBy("location").orderBy(F.col("timestamp_utc").desc())

df_gold_aq_weather = (
    aq_sel.join(weather_sel, on=["location", "timestamp_utc"], how="left")
    .withColumn(
        "aqi_category",
        F.when(F.col("us_aqi") <= 50, F.lit("Baik"))
        .when(F.col("us_aqi") <= 100, F.lit("Sedang"))
        .when(F.col("us_aqi") <= 150, F.lit("Tidak Sehat (Sensitif)"))
        .when(F.col("us_aqi") <= 200, F.lit("Tidak Sehat"))
        .when(F.col("us_aqi") <= 300, F.lit("Sangat Tidak Sehat"))
        .when(F.col("us_aqi") > 300, F.lit("Berbahaya"))
        .otherwise(F.lit("Tidak Diketahui")),
    )
    .withColumn("is_latest_reading", F.row_number().over(w_latest_city) == 1)
    .withColumn("date", F.to_date("timestamp_utc"))
    .withColumn("year", F.year("timestamp_utc"))
    .withColumn("month", F.month("timestamp_utc"))
)

gold_summary["gold_counts"]["dashboard_air_quality_weather"] = write_gold_parquet(
    df_gold_aq_weather,
    table_name="dashboard_air_quality_weather",
    partition_cols=["year", "month"],
)

In [ ]:
# 2. Gold Serving: Status Gunung Api Terkini (MAGMA Indonesia)
w_latest_volcano = Window.partitionBy("volcano_name").orderBy(F.col("scraped_at_utc").desc())

df_gold_volcano = (
    df_magma.withColumn("_rn", F.row_number().over(w_latest_volcano))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn(
        "level_rank",
        F.when(F.col("activity_level").contains("Level IV"), F.lit(4))
        .when(F.col("activity_level").contains("Level III"), F.lit(3))
        .when(F.col("activity_level").contains("Level II"), F.lit(2))
        .when(F.col("activity_level").contains("Level I"), F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("is_high_alert", F.col("level_rank") >= 3)
    .orderBy(F.col("level_rank").desc(), F.col("volcano_name").asc())
)

gold_summary["gold_counts"]["dashboard_volcano_status"] = write_gold_parquet(
    df_gold_volcano,
    table_name="dashboard_volcano_status",
)

In [ ]:
# 3. Gold Serving: Hotspot NASA FIRMS + Irisan Tile MODIS MCD64A1 Burned Scar
mcd_tiles = (
    df_mcd.select(
        F.col("tile_id").alias("mcd_tile_id"),
        "bbox_west",
        "bbox_south",
        "bbox_east",
        "bbox_north",
        F.col("start_datetime").alias("mcd_latest_start"),
    )
    .groupBy("mcd_tile_id", "bbox_west", "bbox_south", "bbox_east", "bbox_north")
    .agg(F.max("mcd_latest_start").alias("mcd_latest_start"))
)

w_hotspot_tile = Window.partitionBy(
    "sensor", "latitude", "longitude", "acq_date", "acq_time"
).orderBy(F.col("mcd_latest_start").desc_nulls_last())

df_gold_hotspots = (
    df_firms.join(
        F.broadcast(mcd_tiles),
        (
            (df_firms["longitude"] >= mcd_tiles["bbox_west"])
            & (df_firms["longitude"] <= mcd_tiles["bbox_east"])
            & (df_firms["latitude"] >= mcd_tiles["bbox_south"])
            & (df_firms["latitude"] <= mcd_tiles["bbox_north"])
        ),
        how="left",
    )
    .withColumn("_rn", F.row_number().over(w_hotspot_tile))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "bbox_west", "bbox_south", "bbox_east", "bbox_north")
    .withColumn("in_burned_scar_tile", F.col("mcd_tile_id").isNotNull())
    .withColumn(
        "fire_intensity_class",
        F.when(F.col("frp") >= 50.0, F.lit("Tinggi"))
        .when(F.col("frp") >= 15.0, F.lit("Sedang"))
        .otherwise(F.lit("Rendah")),
    )
    .withColumn("lat_grid", F.round(F.col("latitude") * 4) / 4)
    .withColumn("lon_grid", F.round(F.col("longitude") * 4) / 4)
)

gold_summary["gold_counts"]["dashboard_fire_hotspots"] = write_gold_parquet(
    df_gold_hotspots,
    table_name="dashboard_fire_hotspots",
    partition_cols=["acq_date"],
)

In [ ]:
# 4. Gold ML Feature Store: Prediksi Risiko Karhutla Harian per Lokasi
df_daily_env = (
    df_gold_aq_weather.groupBy("location", "province", "latitude", "longitude", "date")
    .agg(
        F.avg("temperature_2m").alias("temp_mean"),
        F.max("temperature_2m").alias("temp_max"),
        F.avg("relative_humidity_2m").alias("humidity_mean"),
        F.min("relative_humidity_2m").alias("humidity_min"),
        F.sum("precipitation").alias("precip_sum"),
        F.avg("wind_speed_10m").alias("wind_speed_mean"),
        F.max("wind_speed_10m").alias("wind_speed_max"),
        F.avg("wind_direction_10m").alias("wind_direction_mean"),
        F.avg("surface_pressure").alias("pressure_mean"),
        F.avg("pm2_5").alias("pm2_5_mean"),
        F.max("pm2_5").alias("pm2_5_max"),
        F.avg("pm10").alias("pm10_mean"),
        F.avg("carbon_monoxide").alias("co_mean"),
        F.max("us_aqi").alias("us_aqi_max"),
    )
    .withColumn("is_dry_day", F.when(F.col("precip_sum") < 1.0, F.lit(1)).otherwise(F.lit(0)))
)

locations_dim = df_daily_env.select("location", "latitude", "longitude").distinct()

firms_small = df_firms.select(
    F.col("acq_date").alias("date"),
    F.col("latitude").alias("f_lat"),
    F.col("longitude").alias("f_lon"),
    F.col("frp").alias("f_frp"),
    F.col("brightness").alias("f_brightness"),
)

dlat = F.radians(F.col("f_lat") - F.col("latitude"))
dlon = F.radians(F.col("f_lon") - F.col("longitude"))
a = (
    F.pow(F.sin(dlat / 2), 2)
    + F.cos(F.radians(F.col("latitude")))
    * F.cos(F.radians(F.col("f_lat")))
    * F.pow(F.sin(dlon / 2), 2)
)
dist_km = F.lit(6371.0) * F.lit(2.0) * F.asin(F.sqrt(F.least(F.lit(1.0), a)))

df_daily_hotspots = (
    F.broadcast(locations_dim)
    .crossJoin(firms_small)
    .withColumn("distance_km", dist_km)
    .filter(F.col("distance_km") <= F.lit(float(hotspot_radius_km)))
    .groupBy("location", "date")
    .agg(
        F.count("*").alias("hotspot_count_50km"),
        F.sum("f_frp").alias("frp_sum_50km"),
        F.max("f_brightness").alias("brightness_max_50km"),
    )
)

df_loc_mcd = (
    locations_dim.join(
        F.broadcast(df_mcd),
        (
            (locations_dim["longitude"] >= df_mcd["bbox_west"])
            & (locations_dim["longitude"] <= df_mcd["bbox_east"])
            & (locations_dim["latitude"] >= df_mcd["bbox_south"])
            & (locations_dim["latitude"] <= df_mcd["bbox_north"])
        ),
        how="left",
    )
    .groupBy("location")
    .agg(
        F.countDistinct("item_id").alias("mcd_burned_scar_tiles_count"),
        F.max("start_datetime").alias("mcd_latest_tile_date"),
    )
)

w_loc_time = Window.partitionBy("location").orderBy("date")

df_gold_ml = (
    df_daily_env.join(df_daily_hotspots, on=["location", "date"], how="left")
    .join(df_loc_mcd, on=["location"], how="left")
    .fillna({
        "hotspot_count_50km": 0,
        "frp_sum_50km": 0.0,
        "brightness_max_50km": 0.0,
        "mcd_burned_scar_tiles_count": 0,
    })
    .withColumn("hotspot_lag_1d", F.coalesce(F.lag("hotspot_count_50km", 1).over(w_loc_time), F.lit(0)))
    .withColumn("hotspot_lag_3d", F.coalesce(F.lag("hotspot_count_50km", 3).over(w_loc_time), F.lit(0)))
    .withColumn("frp_lag_1d", F.coalesce(F.lag("frp_sum_50km", 1).over(w_loc_time), F.lit(0.0)))
    .withColumn("dry_days_3d", F.sum("is_dry_day").over(w_loc_time.rowsBetween(-2, 0)))
    .withColumn("pm2_5_3d_avg", F.avg("pm2_5_mean").over(w_loc_time.rowsBetween(-2, 0)))
    .withColumn("has_hotspot_today", F.when(F.col("hotspot_count_50km") > 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("target_hotspot_count_next_day", F.lead("hotspot_count_50km", 1).over(w_loc_time))
    .withColumn(
        "target_fire_risk_next_day",
        F.when(F.col("target_hotspot_count_next_day").isNull(), F.lit(None).cast("int"))
        .when(F.col("target_hotspot_count_next_day") > 0, F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("day_of_year", F.dayofyear("date"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
)

gold_summary["gold_counts"]["ml_karhutla_features"] = write_gold_parquet(
    df_gold_ml,
    table_name="ml_karhutla_features",
    partition_cols=["year", "month"],
)

In [ ]:
# 5. Gold Analytics: KPI Harian Nasional (Tren 7 Hari, DoD Delta, Hotspot & AQI)
fire_daily_nat = (
    df_gold_hotspots.groupBy(F.col("acq_date").alias("date"))
    .agg(
        F.count("*").alias("total_hotspots"),
        F.sum(F.when(F.col("fire_intensity_class") == "Tinggi", 1).otherwise(0)).alias("high_intensity_hotspots"),
        F.sum("frp").alias("total_frp"),
        F.avg("frp").alias("avg_frp"),
        F.sum(F.when(F.col("in_burned_scar_tile"), 1).otherwise(0)).alias("hotspots_in_burned_scar_tile"),
    )
    .withColumn(
        "pct_hotspots_in_burned_scar",
        F.round(F.col("hotspots_in_burned_scar_tile") * 100.0 / F.greatest(F.col("total_hotspots"), F.lit(1)), 2),
    )
)

env_daily_nat = (
    df_daily_env.groupBy("date")
    .agg(
        F.round(F.avg("pm2_5_mean"), 2).alias("national_avg_pm25"),
        F.max("us_aqi_max").alias("national_max_aqi"),
        F.sum(F.when(F.col("us_aqi_max") > 100, 1).otherwise(0)).alias("cities_unhealthy_aqi_count"),
        F.round(F.avg("temp_mean"), 2).alias("national_avg_temp"),
        F.round(F.sum("precip_sum"), 2).alias("national_total_precip"),
    )
)

w_nat_time = Window.orderBy("date")

df_gold_national_kpi = (
    env_daily_nat.join(fire_daily_nat, on="date", how="full_outer")
    .fillna({
        "total_hotspots": 0,
        "high_intensity_hotspots": 0,
        "total_frp": 0.0,
        "avg_frp": 0.0,
        "hotspots_in_burned_scar_tile": 0,
        "pct_hotspots_in_burned_scar": 0.0,
        "cities_unhealthy_aqi_count": 0,
    })
    .withColumn(
        "hotspots_7d_moving_avg",
        F.round(F.avg("total_hotspots").over(w_nat_time.rowsBetween(-6, 0)), 2),
    )
    .withColumn(
        "hotspot_dod_change",
        F.col("total_hotspots") - F.coalesce(F.lag("total_hotspots", 1).over(w_nat_time), F.lit(0)),
    )
    .orderBy("date")
)

gold_summary["gold_counts"]["analytics_daily_national_kpi"] = write_gold_parquet(
    df_gold_national_kpi,
    table_name="analytics_daily_national_kpi",
)

In [ ]:
# 6. Gold Analytics: Scorecard Risiko Multi-Bahaya per Kota/Provinsi (0-100)
fire_score = F.least(
    F.lit(100.0),
    F.col("hotspot_count_50km") * F.lit(15.0)
    + F.col("dry_days_3d") * F.lit(10.0)
    + F.when(F.col("temp_max") >= 33.0, F.lit(15.0)).otherwise(F.lit(0.0))
    + F.when(F.col("humidity_min") <= 55.0, F.lit(15.0)).otherwise(F.lit(0.0))
    + F.when(F.col("mcd_burned_scar_tiles_count") > 0, F.lit(10.0)).otherwise(F.lit(0.0)),
)

aqi_score = F.least(F.lit(100.0), F.coalesce(F.col("us_aqi_max"), F.lit(0.0)) / F.lit(2.0))

df_gold_city_scorecard = (
    df_gold_ml.select(
        "location",
        "province",
        "latitude",
        "longitude",
        "date",
        "temp_max",
        "humidity_min",
        "precip_sum",
        "dry_days_3d",
        "wind_speed_max",
        "wind_direction_mean",
        "pm2_5_max",
        "pm2_5_3d_avg",
        "us_aqi_max",
        "hotspot_count_50km",
        "frp_sum_50km",
        "mcd_burned_scar_tiles_count",
        "year",
        "month",
    )
    .withColumn("fire_risk_score", F.round(fire_score, 1))
    .withColumn("air_quality_risk_score", F.round(aqi_score, 1))
    .withColumn(
        "composite_risk_score",
        F.round(F.col("fire_risk_score") * F.lit(0.6) + F.col("air_quality_risk_score") * F.lit(0.4), 1),
    )
    .withColumn(
        "risk_level",
        F.when(F.col("composite_risk_score") >= 75.0, F.lit("Ekstrem"))
        .when(F.col("composite_risk_score") >= 50.0, F.lit("Tinggi"))
        .when(F.col("composite_risk_score") >= 25.0, F.lit("Sedang"))
        .otherwise(F.lit("Rendah")),
    )
)

gold_summary["gold_counts"]["analytics_city_risk_scorecard"] = write_gold_parquet(
    df_gold_city_scorecard,
    table_name="analytics_city_risk_scorecard",
    partition_cols=["year", "month"],
)

In [ ]:
# 7. Gold Analytics: Ringkasan Aktivitas Gunung Api per Level & Provinsi
df_gold_volcano_summary = (
    df_gold_volcano.groupBy("activity_level", "level_rank", "province")
    .agg(
        F.count("*").alias("volcano_count"),
        F.sum(F.when(F.col("is_high_alert"), 1).otherwise(0)).alias("high_alert_count"),
        F.concat_ws(", ", F.collect_list("volcano_name")).alias("volcano_list"),
        F.max("scraped_at_utc").alias("last_updated_utc"),
    )
    .orderBy(F.col("level_rank").desc(), F.col("volcano_count").desc())
)

gold_summary["gold_counts"]["analytics_volcano_summary"] = write_gold_parquet(
    df_gold_volcano_summary,
    table_name="analytics_volcano_summary",
)

# 8. Gold Analytics: Heatmap Spasial Hotspot per Grid 0.25 Derajat per Bulan
df_gold_spatial_grid = (
    df_gold_hotspots.withColumn("year", F.year("acq_date"))
    .withColumn("month", F.month("acq_date"))
    .groupBy("year", "month", "lat_grid", "lon_grid", "mcd_tile_id", "in_burned_scar_tile")
    .agg(
        F.count("*").alias("hotspot_count"),
        F.round(F.sum("frp"), 2).alias("total_frp"),
        F.round(F.max("frp"), 2).alias("max_frp"),
        F.sum(F.when(F.col("fire_intensity_class") == "Tinggi", 1).otherwise(0)).alias("high_intensity_count"),
        F.countDistinct("acq_date").alias("active_fire_days"),
    )
    .orderBy(F.col("hotspot_count").desc())
)

gold_summary["gold_counts"]["analytics_spatial_hotspot_grid"] = write_gold_parquet(
    df_gold_spatial_grid,
    table_name="analytics_spatial_hotspot_grid",
    partition_cols=["year", "month"],
)

In [ ]:
print(json.dumps(gold_summary, indent=2))
mssparkutils.notebook.exit(json.dumps(gold_summary))